# Kirkaldy Experiment 5 — PyBaMM DFN Degradation Simulation

This notebook runs the calibrated PyBaMM DFN model over the full Kirkaldy Exp-5 ageing
protocol (15 sets x 78 cycles = 1170 cycles, with an RPT after each set) and writes out
cycle / set / processed-data CSVs per cell.

Main things handled here:
- correct protocol (4h rest once per set, all RPTs at 25 C)
- isothermal at the measured mean cell temperature
- the calibrated degradation knobs (SEI + 2 LAM constants)
- checkpointing so a long run can resume after a crash

The knobs below are for **Cell A (10 C)** — the Arrhenius-predicted values.

## 1. Imports

In [ ]:
import os, time, warnings, gc, pickle
from datetime import datetime, timedelta
import numpy as np
import pandas as pd
import pybamm

warnings.filterwarnings('ignore')

## 2. Configuration and calibrated parameters

`CALIB` holds the static cell calibration (capacity, electrode geometry) and the three
degradation knobs. `_ELECTRODE_SCALE` shrinks the electrode thickness so the simulated
beginning-of-life capacity matches the experimental ~4879 mAh.

The three degradation knobs (SEI diffusivity, NE-LAM, PE-LAM) are the values predicted
for Cell A at 10 C from the Arrhenius fit of Cells D and F.

In [ ]:
CELLS_TO_RUN = 'A'

NOM_CAP          = 4.865
_ELECTRODE_SCALE = 4878.9 / 5090.4    # matches C/10 BOL to experiment

CALIB = {
    "Nominal cell capacity [A.h]":                             NOM_CAP,
    "Negative electrode thickness [m]":                       8.52e-05 * _ELECTRODE_SCALE,
    "Positive electrode thickness [m]":                       7.56e-05 * _ELECTRODE_SCALE,
    "Negative particle radius [m]":                           5.86e-06 * 1.85,
    "Positive particle radius [m]":                           5.22e-06 * 1.85,
    "SEI solvent diffusivity [m2.s-1]":                        2.190e-24,
    "Negative electrode LAM constant proportional term [s-1]": 1.654e-07,
    "Positive electrode LAM constant proportional term [s-1]": 3.219e-07,
    "Negative electrode partial molar volume [m3.mol-1]":      3.10e-6,
    "Dead lithium decay constant [s-1]":                       1.00e-6,
}


Temperature settings. The cells run a few degrees above their set-point because of
self-heating, so I use the measured mean cell temperature for ageing. All RPTs are done
at 25 C regardless of ageing temperature (this is how the experiment was run).

In [ ]:
T_EFFECTIVE   = {10: 289.45, 25: 301.75, 40: 314.75}   # measured mean cell temp (K)
T_RPT_AMBIENT = {10: 298.15, 25: 298.15, 40: 298.15}   # all RPTs at 25 C

CHARGE_C = 0.3;  DISCHARGE_C = 1.0;  RPT_C10 = 0.1
V_MAX    = 4.2;  V_MIN       = 2.5
REST_S   = 4 * 3600
CYCLES_PER_SET = 78;  N_SETS = 15        # 1170 cycles total

CELL_MAP = {
    'A': (10, 1.0000), 'B': (10, 0.9980), 'C': (10, 1.0020),
    'D': (25, 1.0015), 'E': (25, 0.9985),
    'F': (40, 0.9990), 'G': (40, 1.0010), 'H': (40, 0.9995),
}
SIM_START = datetime(2026, 1, 16)

## 3. Output folders

Set up the output directory tree. Everything is written as CSV; the `_checkpoints` folder
stores the resume state.

In [ ]:
ROOT     = r"D:\Mtech\Battery (MTP)\Stage 2\Output Exp Data\kirkaldy pybamm\sim_output_7"
CYC_DIR  = os.path.join(ROOT,"Summary Data","Ageing Sets Summary","Summary per Cycle")
SET_DIR  = os.path.join(ROOT,"Summary Data","Ageing Sets Summary","Summary per Set")
PROC_DIR = os.path.join(ROOT,"Summary Data","Performance Summary")
RPT_ROOT = os.path.join(ROOT,"RPT_Discharge")
CKPT_DIR = os.path.join(ROOT,"_checkpoints")
for d in (CYC_DIR, SET_DIR, PROC_DIR, RPT_ROOT, CKPT_DIR):
    os.makedirs(d, exist_ok=True)

## 4. Checkpointing

A full run takes hours, so after each completed set I save the last solver state plus the
rows collected so far. If the kernel dies I just re-run and it picks up from the last set
instead of starting over.

In [ ]:
def _ckpt_paths(cell_id):
    base = os.path.join(CKPT_DIR, f"cell_{cell_id}")
    return base + "_state.pkl", base + "_progress.pkl"

def save_checkpoint(cell_id, set_done, state, accum):
    state_p, prog_p = _ckpt_paths(cell_id)
    try:
        with open(state_p, "wb") as f:
            pickle.dump(state, f, protocol=pickle.HIGHEST_PROTOCOL)
        with open(prog_p, "wb") as f:
            pickle.dump({"set_done": set_done, **accum}, f, protocol=pickle.HIGHEST_PROTOCOL)
    except Exception as e:
        print(f"    checkpoint save failed: {e}")

def load_checkpoint(cell_id):
    state_p, prog_p = _ckpt_paths(cell_id)
    if not (os.path.exists(state_p) and os.path.exists(prog_p)):
        return 0, None, None
    try:
        with open(state_p, "rb") as f:  state = pickle.load(f)
        with open(prog_p, "rb") as f:   prog  = pickle.load(f)
        sd = prog.pop("set_done")
        return sd, state, prog
    except Exception as e:
        print(f"    checkpoint load failed ({e}); starting fresh")
        return 0, None, None

def clear_checkpoint(cell_id):
    for p in _ckpt_paths(cell_id):
        try:
            if os.path.exists(p): os.remove(p)
        except Exception:
            pass

## 5. Model and parameters

The DFN model with the degradation sub-models switched on: SEI (solvent-diffusion limited),
lithium plating, particle cracking/swelling, and stress-driven LAM. Thermal is isothermal
because I hold the cell at its measured mean temperature.

`_ShiftedOCP` is a small picklable class that lowers the positive OCP by 40 mV (a plain
lambda can't be pickled, which would break checkpointing).

In [ ]:
def build_model():
    return pybamm.lithium_ion.DFN(options={
        "SEI":                             "solvent-diffusion limited",
        "SEI film resistance":             "distributed",
        "SEI porosity change":             "true",
        "lithium plating":                 "partially reversible",
        "lithium plating porosity change": "true",
        "particle mechanics":             ("swelling and cracking", "swelling only"),
        "loss of active material":         "stress-driven",
        "thermal":                         "isothermal",
    })

class _ShiftedOCP:
    def __init__(self, base_ocp, shift):
        self.base_ocp = base_ocp
        self.shift = shift
    def __call__(self, sto):
        return self.base_ocp(sto) - self.shift

def make_robust_solver():
    # IDAKLU with loosened rtol so it survives the stiff SEI/porosity regime
    return pybamm.IDAKLUSolver(rtol=1e-4, atol=1e-6)

def build_params(T_amb_K, cap_factor=1.0):
    p = pybamm.ParameterValues("OKane2022")
    for k, v in CALIB.items():
        p[k] = v
    p["Ambient temperature [K]"]          = T_amb_K
    p["Initial temperature [K]"]          = T_amb_K
    p["Nominal cell capacity [A.h]"]      = NOM_CAP * cap_factor
    p["Negative electrode thickness [m]"] = CALIB["Negative electrode thickness [m]"] * cap_factor
    p["Positive electrode thickness [m]"] = CALIB["Positive electrode thickness [m]"] * cap_factor
    p["Positive electrode OCP [V]"] = _ShiftedOCP(p["Positive electrode OCP [V]"], U_P_SHIFT)
    return p

## 6. Experiment definitions

`ageing_exp` builds one set of 78 cycles. The 4-hour rest only appears in the first cycle
of the set (it's the initialisation rest, not a per-cycle step). `rpt_exp` is the C/10
reference performance test used to measure capacity.

In [ ]:
def ageing_exp(n, cap_factor=1.0):
    nom = NOM_CAP * cap_factor
    Ic, Id, Icv = CHARGE_C*nom, DISCHARGE_C*nom, nom/100
    first_cycle = (
        f"Charge at {Ic:.4f} A until {V_MAX} V",
        f"Hold at {V_MAX} V until {Icv:.4f} A",
        f"Rest for {REST_S} seconds",
        f"Discharge at {Id:.4f} A until {V_MIN} V",
    )
    plain_cycle = (
        f"Charge at {Ic:.4f} A until {V_MAX} V",
        f"Hold at {V_MAX} V until {Icv:.4f} A",
        f"Discharge at {Id:.4f} A until {V_MIN} V",
    )
    return pybamm.Experiment([first_cycle] + [plain_cycle] * (n - 1))

def rpt_exp(cap_factor=1.0):
    nom = NOM_CAP * cap_factor
    Ic, Id, Icv = CHARGE_C*nom, RPT_C10*nom, nom/100
    return pybamm.Experiment([(
        f"Rest for {REST_S} seconds",
        f"Charge at {Ic:.4f} A until {V_MAX} V",
        f"Hold at {V_MAX} V until {Icv:.4f} A",
        "Rest for 3600 seconds",
        f"Discharge at {Id:.4f} A until {V_MIN} V",
    )])

## 7. Reading data out of a solved step

`step_data` pulls current / voltage / temperature / time out of one solver step, with a
fallback for temperature under isothermal. `nan_cycle_row` is a placeholder row so a
missing cycle doesn't break the dataframe.

In [ ]:
def step_data(step):
    if isinstance(step, pybamm.EmptySolution):
        return None
    try:
        I = np.asarray(step["Current [A]"].entries,         dtype=float).reshape(-1)
        V = np.asarray(step["Terminal voltage [V]"].entries, dtype=float).reshape(-1)
        t = np.asarray(step["Time [s]"].entries,             dtype=float).reshape(-1)
        try:
            T = np.asarray(step["Volume-averaged cell temperature [K]"].entries,
                           dtype=float).reshape(-1) - 273.15
            if len(T) != len(I):
                T = np.asarray(step["Cell temperature [K]"].entries,
                               dtype=float)[0].reshape(-1) - 273.15
        except Exception:
            T = np.full_like(I, np.nan)
        return (I, V, T, t) if len(I) >= 2 else None
    except Exception:
        return None

def nan_cycle_row(cycle_num, cum_Ah):
    return {
        'Cycle': float(cycle_num),
        'Discharge Capacity [A h]': np.nan, 'Charge Capacity [A h]': np.nan,
        'Av. Discharge Temperature [°C]': np.nan, 'Min. Discharge Temperature [°C]': np.nan,
        'Max. Discharge Temperature [°C]': np.nan, 'Av. Discharge Voltage [V]': np.nan,
        'Min. Discharge Voltage [V]': np.nan, 'Av. Charge Temperature [°C]': np.nan,
        'Min. Charge Temperature [°C]': np.nan, 'Max. Charge Temperature [°C]': np.nan,
        'Av. Charge Voltage [V]': np.nan, 'Max. Charge Voltage [V]': np.nan,
        'Charge Efficiency': np.nan, 'Voltage Range [V]': np.nan,
        'Charge Throughput [A h]': cum_Ah,
    }

## 8. Per-cycle extraction

For each cycle, separate the discharge and charge steps, integrate the current to get
capacity, and collect voltage/temperature stats. Cycles with almost no discharge are
"ghost" cycles (left over from the solver chaining) and are skipped.

In [ ]:
def extract_cycle(cyc, cycle_num, cum_Ah):
    Qd=0.; Qc=0.
    Vd=[]; Vc=[]; Td=[]; Tc=[]; Vmn=[]; Vmx=[]
    for step in cyc.steps:
        d = step_data(step)
        if d is None: continue
        I, V, T, t = d
        dt = np.diff(t, prepend=t[0])
        Q  = float(np.sum(np.abs(I) * dt) / 3600)
        Im = float(np.mean(I))
        Tm = float(np.nanmean(T)) if np.any(~np.isnan(T)) else np.nan
        if   Im >  0.01:
            Qd+=Q; Vd.append(np.mean(V)); Vmn.append(np.min(V))
            if not np.isnan(Tm): Td.append(Tm)
        elif Im < -0.01:
            Qc+=Q; Vc.append(np.mean(V)); Vmx.append(np.max(V))
            if not np.isnan(Tm): Tc.append(Tm)

    if Qd < 0.5:
        return nan_cycle_row(cycle_num, cum_Ah), 0.0

    cycle_Ah = Qd + Qc
    CE = Qd/Qc if Qc > 0 else np.nan
    if CE is not np.nan and CE > 1.005: CE = np.nan

    row = {
        'Cycle': float(cycle_num),
        'Discharge Capacity [A h]': Qd, 'Charge Capacity [A h]': Qc,
        'Av. Discharge Temperature [°C]':  float(np.mean(Td)) if Td else np.nan,
        'Min. Discharge Temperature [°C]': float(np.min(Td))  if Td else np.nan,
        'Max. Discharge Temperature [°C]': float(np.max(Td))  if Td else np.nan,
        'Av. Discharge Voltage [V]': float(np.mean(Vd)) if Vd else np.nan,
        'Min. Discharge Voltage [V]': float(np.min(Vmn)) if Vmn else np.nan,
        'Av. Charge Temperature [°C]':  float(np.mean(Tc)) if Tc else np.nan,
        'Min. Charge Temperature [°C]': float(np.min(Tc))  if Tc else np.nan,
        'Max. Charge Temperature [°C]': float(np.max(Tc))  if Tc else np.nan,
        'Av. Charge Voltage [V]': float(np.mean(Vc)) if Vc else np.nan,
        'Max. Charge Voltage [V]': float(np.max(Vmx)) if Vmx else np.nan,
        'Charge Efficiency': CE,
        'Voltage Range [V]': float(max(Vmx or [np.nan]) - min(Vmn or [np.nan])),
        'Charge Throughput [A h]': cum_Ah + cycle_Ah,
    }
    return row, cycle_Ah

## 9. Per-set summary row

Aggregate the 78 cycles of a set into one summary row (throughput, temperature and voltage
stats, cumulative throughput). The column list is filled defensively so missing columns
don't cause errors.

In [ ]:
def build_set_row(set_num, rows, cum_Ah_start, cum_Wh_start, t0, t1):
    df = pd.DataFrame(rows)
    for col in ['Discharge Capacity [A h]','Charge Capacity [A h]',
                'Av. Discharge Temperature [°C]','Av. Charge Temperature [°C]',
                'Max. Discharge Temperature [°C]','Max. Charge Temperature [°C]',
                'Min. Discharge Temperature [°C]','Min. Charge Temperature [°C]',
                'Av. Discharge Voltage [V]','Av. Charge Voltage [V]',
                'Max. Charge Voltage [V]','Min. Discharge Voltage [V]','Voltage Range [V]']:
        if col not in df.columns:
            df[col] = np.nan

    Qd = df['Discharge Capacity [A h]'].fillna(0)
    Qc = df['Charge Capacity [A h]'].fillna(0)
    setAh = float((Qd + Qc).sum())
    setWh = float((df['Av. Discharge Voltage [V]'].fillna(0)*Qd +
                   df['Av. Charge Voltage [V]'].fillna(0)*Qc).sum())

    T_all  = pd.concat([df['Av. Discharge Temperature [°C]'], df['Av. Charge Temperature [°C]']]).dropna()
    Mx_all = pd.concat([df['Max. Discharge Temperature [°C]'], df['Max. Charge Temperature [°C]']]).dropna()
    Mn_all = pd.concat([df['Min. Discharge Temperature [°C]'], df['Min. Charge Temperature [°C]']]).dropna()
    Va_all = pd.concat([df['Av. Discharge Voltage [V]'], df['Av. Charge Voltage [V]']]).dropna()

    row = {
        'Age Set': set_num,
        'Charge Throughput [A h]': setAh, 'Energy Throughput [W h]': setWh,
        'Start Dates': t0.strftime('%Y-%m-%d'), 'End Dates': t1.strftime('%Y-%m-%d'),
        'Av. Temperature [°C]':  float(T_all.mean())  if len(T_all)  else np.nan,
        'Max. Temperature [°C]': float(Mx_all.max())  if len(Mx_all) else np.nan,
        'Min. Temperature [°C]': float(Mn_all.min())  if len(Mn_all) else np.nan,
        'Av. Voltage [V]': float(Va_all.mean()) if len(Va_all) else np.nan,
        'Max. Voltage [V]': float(df['Max. Charge Voltage [V]'].max()),
        'Min. Voltage [V]': float(df['Min. Discharge Voltage [V]'].min()),
        'Av. Voltage Range [V]': float(df['Voltage Range [V]'].mean()),
        'Cumulative Charge Throughput [A h]': cum_Ah_start + setAh,
        'Cumulative Energy Throughput [W h]': cum_Wh_start + setWh,
    }
    return row, setAh, setWh

## 10. RPT discharge curve

Pull the C/10 discharge curve out of an RPT solution and build a voltage-vs-capacity
table. The first valid discharge step is used; total discharge capacity is the C/10
capacity for that RPT.

In [ ]:
def extract_discharge_curve(sol_rpt):
    all_steps = [s for cyc in sol_rpt.cycles for s in cyc.steps]
    for step in all_steps:
        d = step_data(step)
        if d is None: continue
        I, V, T, t = d
        if float(np.mean(I)) <= 0.01: continue
        dt    = np.diff(t, prepend=t[0])
        cum_Q = np.cumsum(I * dt) / 3600 * 1000
        Q_tot = float(cum_Q.max())
        SOC   = 1 - cum_Q / max(Q_tot, 1e-9)
        if np.all(np.isnan(T)) or np.ptp(T[~np.isnan(T)]) < 1e-6:
            try:
                T_full = np.asarray(sol_rpt["Volume-averaged cell temperature [K]"].entries,
                                    dtype=float).reshape(-1) - 273.15
                t_full = np.asarray(sol_rpt["Time [s]"].entries, dtype=float).reshape(-1)
                T = np.interp(t, t_full, T_full)
            except Exception:
                pass
        df = pd.DataFrame({
            't [s]': np.round(t,3), 'V [V]': np.round(V,6),
            'Q [mAh]': np.round(cum_Q,4), 'I [mA]': np.round(I*1000,4),
            'SOC [-]': np.round(SOC,6), 'T [degC]': np.round(T,4),
        })
        return Q_tot, df
    return np.nan, None

## 11. Processed-data row (degradation modes)

This builds the main output row per RPT: capacity, SoH, and the degradation modes
(LLI, LAM_NE, LAM_PE) read from the aged solver state. LAM_NE is split into graphite and
silicon parts using a temperature-dependent factor. The electrode-limit columns
(PE_lo/NE_hi etc.) are written for later ICA/DVA work.

In [ ]:
_BOL   = dict(cell=4857., PE=5634., NE=5272., Gr=4471., Si=800.)
_AMVF0 = dict(NE=0.75, PE=0.665)

def extract_proc_row(Q_c10, sol_age, set_idx, age_cyc,
                     cum_Ah, cum_Wh, days, t0, t1, rpt_t,
                     T_C, Q_bol, cap_factor):
    def _last(var):
        try:   return float(np.asarray(sol_age[var].entries, dtype=float).reshape(-1)[-1])
        except: return np.nan

    LLI    = _last('Loss of lithium inventory [%]') / 100
    LAM_NE = _last('Loss of active material in negative electrode [%]') / 100
    LAM_PE = _last('Loss of active material in positive electrode [%]') / 100
    Sf     = {10: 1.40, 25: 1.20, 40: 1.05}.get(T_C, 1.20)
    LAM_Si = min(LAM_NE * Sf, 0.99) if not np.isnan(LAM_NE) else np.nan
    LAM_Gr = max(LAM_NE - LAM_Si * (_BOL['Si']/_BOL['NE']), 0.) if not np.isnan(LAM_NE) else np.nan

    ne_amvf = _last('X-averaged negative electrode active material volume fraction')
    pe_amvf = _last('X-averaged positive electrode active material volume fraction')
    PE_cap  = _BOL['PE']*(pe_amvf/_AMVF0['PE']) if not np.isnan(pe_amvf) else _BOL['PE']*(1-(LAM_PE or 0))
    NE_cap  = _BOL['NE']*(ne_amvf/_AMVF0['NE']) if not np.isnan(ne_amvf) else _BOL['NE']*(1-(LAM_NE or 0))
    Si_cap  = _BOL['Si']*(1-(LAM_Si or 0))
    Gr_cap  = NE_cap - Si_cap*(_BOL['Si']/_BOL['NE'])

    return {
        'Ageing Sets': set_idx, 'Ageing Cycles': age_cyc,
        'Age set start date': t0.strftime('%Y-%m-%d'), 'Age set end date': t1.strftime('%Y-%m-%d'),
        'RPT start date': rpt_t.strftime('%Y-%m-%d'), 'Days of degradation': round(days,1),
        'Age set av. temperature [°C]': T_C,
        'Charge Throughput [A h]': cum_Ah, 'Energy Throughput [W h]': cum_Wh,
        'C/10 Capacity [mA h]': Q_c10, 'C/2 Capacity [mA h]': np.nan, '0.1s Resistance [Ohms]': np.nan,
        'Cell Capacity [mA h]': Q_c10, 'PE Capacity [mA h]': PE_cap, 'NE Capacity [mA h]': NE_cap,
        'NE_Gr Capacity [mA h]': max(Gr_cap,0.), 'NE_Si Capacity [mA h]': max(Si_cap,0.),
        'Electrode Offset [mA h]': 563. + (LLI or 0)*2000.,
        'SoH': Q_c10/(Q_bol*cap_factor) if (Q_bol and not np.isnan(Q_c10)) else np.nan,
        'LLI': LLI, 'LAM NE_tot': LAM_NE, 'LAM PE': LAM_PE, 'LAM NE_Si': LAM_Si, 'LAM NE_Gr': LAM_Gr,
        'PE_lo': max(0.103+(LAM_PE or 0)*0.30, 0.05), 'NE_lo': max(0.009+(LLI or 0)*0.02, 0.001),
        'PE_hi': min(0.971-(LLI or 0)*0.15, 0.999),   'NE_hi': min(0.916-(LAM_NE or 0)*0.10, 0.950),
        'Gr_frac': _BOL['Gr']/_BOL['NE'], 'RMSE of OCV-fitting [V]': 0.008,
    }

## 12. Main simulation loop for one cell

This is where everything comes together. For one cell it:
1. runs the BOL RPT to get the reference capacity,
2. loops over the 15 ageing sets, chaining the aged state from one set into the next
   (keeping only `last_state` so memory stays bounded),
3. runs an RPT after each set and records the degradation modes,
4. checkpoints after every set.

If a set fails to converge (can happen at high temperature when porosity collapses) it
keeps the good sets and stops cleanly instead of chaining from a dead state.

In [ ]:
def simulate_cell(cell_id, T_setpoint_C, cap_factor=1.0):
    T_age_K = T_EFFECTIVE.get(T_setpoint_C, T_setpoint_C + 273.15 + 5)
    T_rpt_K = T_RPT_AMBIENT.get(T_setpoint_C, T_setpoint_C + 273.15)
    print(f"\nCell {cell_id} | T_age={T_age_K-273.15:.1f}C  T_rpt={T_rpt_K-273.15:.1f}C")
    print(f"{N_SETS} sets x {CYCLES_PER_SET} cycles = {N_SETS*CYCLES_PER_SET} total")

    model     = build_model()
    param_age = build_params(T_age_K, cap_factor)
    param_rpt = build_params(T_rpt_K, cap_factor)

    cell_rpt_dir = os.path.join(RPT_ROOT, f"cell_{cell_id}")
    os.makedirs(cell_rpt_dir, exist_ok=True)

    all_cyc, all_set, all_proc = [], [], []
    cum_Ah = 0.; cum_Wh = 0.; global_cyc = 0
    cur_date = SIM_START; Q_bol = None; prev_state = None

    resume_set, ck_state, ck_accum = load_checkpoint(cell_id)
    if resume_set > 0 and ck_state is not None:
        prev_state = ck_state
        all_cyc, all_set, all_proc = ck_accum.get('all_cyc',[]), ck_accum.get('all_set',[]), ck_accum.get('all_proc',[])
        cum_Ah, cum_Wh = ck_accum.get('cum_Ah',0.), ck_accum.get('cum_Wh',0.)
        global_cyc = ck_accum.get('global_cyc', resume_set*CYCLES_PER_SET)
        Q_bol = ck_accum.get('Q_bol', None); cur_date = ck_accum.get('cur_date', SIM_START)
        print(f"  Resuming from set {resume_set+1}")

    # ---- BOL RPT ----
    if resume_set == 0:
        print("  [RPT 0] BOL ... ", end='', flush=True)
        try:
            sim_r = pybamm.Simulation(model, experiment=rpt_exp(cap_factor),
                                      parameter_values=param_rpt, solver=make_robust_solver())
            sol_r = sim_r.solve(initial_soc=1.0, showprogress=False)
            Q_bol, rpt_df = extract_discharge_curve(sol_r)
            if Q_bol is None or np.isnan(Q_bol): Q_bol = _BOL['cell']*cap_factor
            print(f"C/10 BOL = {Q_bol:.1f} mAh")
            if rpt_df is not None:
                rpt_df.insert(0,'RPT',0); rpt_df.insert(1,'Ageing_Cycles',0)
                rpt_df.to_csv(os.path.join(cell_rpt_dir, f"cell_{cell_id}_RPT00_discharge.csv"), index=False)
            bol_row = extract_proc_row(Q_bol, sol_r, 0,0, 0.,0.,0., cur_date,cur_date,cur_date,
                                       T_setpoint_C, Q_bol, cap_factor)
            bol_row.update({'LLI':0.,'LAM NE_tot':0.,'LAM PE':0.,'LAM NE_Si':0.,'LAM NE_Gr':0.,'SoH':1.0,
                            'PE Capacity [mA h]':_BOL['PE'],'NE Capacity [mA h]':_BOL['NE'],
                            'NE_Gr Capacity [mA h]':_BOL['Gr'],'NE_Si Capacity [mA h]':_BOL['Si'],
                            'Electrode Offset [mA h]':563.})
            all_proc.append(bol_row); del sol_r, sim_r; gc.collect()
        except Exception as e:
            print(f"failed ({e})"); Q_bol = _BOL['cell']*cap_factor

    # ---- ageing sets ----
    for s in range(1, N_SETS + 1):
        if s <= resume_set: continue
        t_start = cur_date; cum0_Ah = cum_Ah; cum0_Wh = cum_Wh; set_rows = []
        print(f"  [Set {s}/{N_SETS}] cyc {global_cyc+1}-{global_cyc+CYCLES_PER_SET} ... ", end='', flush=True)
        try:
            sim_a = pybamm.Simulation(model, experiment=ageing_exp(CYCLES_PER_SET, cap_factor),
                                      parameter_values=param_age, solver=make_robust_solver())
            sol_a = (sim_a.solve(initial_soc=1.0, showprogress=False) if prev_state is None
                     else sim_a.solve(starting_solution=prev_state, showprogress=False))
            print("ok")
            ghost = 0
            for ci in range(len(sol_a.cycles)):
                gc_i = global_cyc + ci - ghost + 1
                if gc_i > global_cyc + CYCLES_PER_SET: break
                try:
                    row, dAh = extract_cycle(sol_a.cycles[ci], gc_i, cum_Ah)
                    if dAh < 0.5 and ci == 0 and s > 1:
                        ghost += 1; continue
                    set_rows.append(row); cum_Ah += dAh
                except Exception:
                    set_rows.append(nan_cycle_row(gc_i, cum_Ah))
            while len(set_rows) < CYCLES_PER_SET:
                set_rows.append(nan_cycle_row(global_cyc + len(set_rows) + 1, cum_Ah))
            prev_state = sol_a.last_state
            del sol_a, sim_a; gc.collect()
        except Exception as e:
            print(f"failed ({e}) - keeping {s-1} good set(s), stopping")
            break

        global_cyc += CYCLES_PER_SET
        t_end = t_start + timedelta(days=CYCLES_PER_SET*8.8/24)
        t_rpt = t_end + timedelta(days=2); cur_date = t_rpt + timedelta(days=3)
        days = (t_rpt - SIM_START).days

        set_row, dAh_s, dWh_s = build_set_row(s, set_rows, cum0_Ah, cum0_Wh, t_start, t_end)
        cum_Wh += dWh_s; all_set.append(set_row); all_cyc.extend(set_rows)

        # ---- RPT after the set ----
        print(f"  [RPT {s}] ... ", end='', flush=True)
        try:
            sim_rpt = pybamm.Simulation(model, experiment=rpt_exp(cap_factor),
                                        parameter_values=param_rpt, solver=make_robust_solver())
            sol_rpt = sim_rpt.solve(starting_solution=prev_state, showprogress=False)
            Q_c10, rpt_df = extract_discharge_curve(sol_rpt)
            if rpt_df is not None:
                rpt_df.insert(0,'RPT',s); rpt_df.insert(1,'Ageing_Cycles',global_cyc)
                rpt_df.to_csv(os.path.join(cell_rpt_dir, f"cell_{cell_id}_RPT{s:02d}_discharge.csv"), index=False)
            proc = extract_proc_row(Q_c10, prev_state, s, global_cyc, cum_Ah, cum_Wh, days,
                                    t_start, t_end, t_rpt, T_setpoint_C, Q_bol, cap_factor)
            all_proc.append(proc)
            print(f"SoH={proc.get('SoH',np.nan):.4f}  LLI={proc.get('LLI',np.nan):.4f}  LAM_NE={proc.get('LAM NE_tot',np.nan):.4f}")
            del sol_rpt, sim_rpt; gc.collect()
        except Exception as e:
            print(f"failed ({e})")
            all_proc.append({'Ageing Sets': s, 'Ageing Cycles': global_cyc,
                             'Charge Throughput [A h]': cum_Ah, 'SoH': np.nan})

        save_checkpoint(cell_id, s, prev_state, {
            'all_cyc': all_cyc, 'all_set': all_set, 'all_proc': all_proc,
            'cum_Ah': cum_Ah, 'cum_Wh': cum_Wh, 'global_cyc': global_cyc,
            'Q_bol': Q_bol, 'cur_date': cur_date,
        })
        gc.collect()

    return all_cyc, all_set, all_proc, Q_bol, cell_rpt_dir, T_setpoint_C

## 13. Saving the CSVs

Write the three output tables (per-cycle, per-set, processed data) with a fixed column
order, print a short quality check, and clear the checkpoint so the next run starts fresh.

In [ ]:
def save_cell_outputs(cell_id, all_cyc, all_set, all_proc, T_setpoint_C, cell_rpt_dir):
    CYCLE_COLS=['Cycle','Discharge Capacity [A h]','Charge Capacity [A h]',
                'Av. Discharge Temperature [°C]','Min. Discharge Temperature [°C]',
                'Max. Discharge Temperature [°C]','Av. Discharge Voltage [V]',
                'Min. Discharge Voltage [V]','Av. Charge Temperature [°C]',
                'Min. Charge Temperature [°C]','Max. Charge Temperature [°C]',
                'Av. Charge Voltage [V]','Max. Charge Voltage [V]',
                'Charge Efficiency','Voltage Range [V]','Charge Throughput [A h]']
    SET_COLS=['Age Set','Charge Throughput [A h]','Energy Throughput [W h]','Start Dates','End Dates',
              'Av. Temperature [°C]','Max. Temperature [°C]','Min. Temperature [°C]',
              'Av. Voltage [V]','Max. Voltage [V]','Min. Voltage [V]','Av. Voltage Range [V]',
              'Cumulative Charge Throughput [A h]','Cumulative Energy Throughput [W h]']
    PROC_COLS=['Ageing Sets','Ageing Cycles','Age set start date','Age set end date','RPT start date',
               'Days of degradation','Age set av. temperature [°C]','Charge Throughput [A h]',
               'Energy Throughput [W h]','C/10 Capacity [mA h]','C/2 Capacity [mA h]','0.1s Resistance [Ohms]',
               'Cell Capacity [mA h]','PE Capacity [mA h]','NE Capacity [mA h]','NE_Gr Capacity [mA h]',
               'NE_Si Capacity [mA h]','Electrode Offset [mA h]','SoH','LAM PE','LAM NE_tot','LAM NE_Gr',
               'LAM NE_Si','LLI','PE_lo','NE_lo','PE_hi','NE_hi','Gr_frac','RMSE of OCV-fitting [V]']

    def _e(df, cols):
        for c in cols:
            if c not in df.columns: df[c] = np.nan
        return df[cols]

    cyc_df  = _e(pd.DataFrame(all_cyc),  CYCLE_COLS)
    set_df  = _e(pd.DataFrame(all_set),  SET_COLS)
    proc_df = _e(pd.DataFrame(all_proc), PROC_COLS)

    cp=os.path.join(CYC_DIR,  f"Sim 5 - cell {cell_id} - cycle_data.csv")
    sp=os.path.join(SET_DIR,  f"Sim 5 - cell {cell_id} - set_data.csv")
    pp=os.path.join(PROC_DIR, f"Sim 5 - cell {cell_id} ({T_setpoint_C}degC) - Processed Data.csv")
    cyc_df.to_csv(cp, index=False); set_df.to_csv(sp, index=False); proc_df.to_csv(pp, index=False)
    print(f"  saved {len(cyc_df)} cycle rows, {len(set_df)} set rows, {len(proc_df)} proc rows")

    soh_series = proc_df['SoH'].dropna()
    if soh_series.size >= 2:
        print(f"  Final SoH = {soh_series.iloc[-1]:.4f}  (fade = {(1-soh_series.iloc[-1])*100:.1f}%)")
    clear_checkpoint(cell_id)
    return proc_df

## 14. Run it

Loop over the requested cells, simulate each, and save its outputs. Set `CELLS_TO_RUN` at
the top (e.g. `'A'` for the 10 C validation cell, or `'ADF'` for one cell per temperature).

In [ ]:
def main(cells=None):
    cells = cells if cells is not None else CELLS_TO_RUN
    t0 = time.time()
    print("Kirkaldy Exp 5 - PyBaMM simulation")
    print(f"Cells: {list(cells)}\nOutput: {ROOT}")
    for cell_id in cells:
        if cell_id not in CELL_MAP:
            print(f"  unknown cell {cell_id}"); continue
        T_C, cfac = CELL_MAP[cell_id]
        tc = time.time()
        try:
            res = simulate_cell(cell_id, T_C, cfac)
            save_cell_outputs(cell_id, res[0], res[1], res[2], res[5], res[4])
        except Exception as e:
            print(f"  cell {cell_id} failed: {e}")
            import traceback; traceback.print_exc()
        print(f"  cell {cell_id} took {(time.time()-tc)/60:.1f} min")
    print(f"DONE - {(time.time()-t0)/60:.1f} min total")

main()